# Notebook 3: Angle Estimation
**Goal:** For each detected tube lid, estimate the rotation angle (joint → tab direction) from the bounding box crops.

**Method:** This notebook utilizes a contour shape-analysis approach (`contour_deviation_largest_component`). Instead of relying on raw pixel brightness—which is vulnerable to crescent glare from overhead lighting—this method binarizes the bounding box crop to extract continuous contours. By isolating the largest connected component (representing the solid plastic lid and tab), the algorithm maps the physical boundary of the object. The tab is then located by analyzing the geometric deviation or protrusion from the expected circular body, allowing for a robust trigonometric calculation of the angle.

**Key note on angle convention:**
- 0° = +X (rightward), angles increase counter-clockwise.
- In image coordinates Y increases downward, so: `dy_image = -sin(angle_rad)`
- The tab is typically a small protrusion on one side of the lid — you need to identify the direction FROM the joint TO the tab.

In [ ]:
import os, cv2, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# ── Paths (same as rest of notebook) ─────────────────────────────────────────
DATA_ROOT = '/content/dataset'
IMG_DIR   = f'{DATA_ROOT}/images'
ANN_CSV   = f'{DATA_ROOT}/annotations.csv'

ann = pd.read_csv(ANN_CSV)
all_images = ann['image'].unique()
_, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)
val_set = set(val_imgs)
gt_val  = ann[ann['image'].isin(val_set)].reset_index(drop=True)

def circ_err(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)

In [ ]:
def angle_clean_contour(img_bgr, cx, cy, bx, by, bw, bh,
                         outlier_thresh=1.05):
    """
    Contour deviation on the LARGEST CONNECTED COMPONENT only.

    Steps:
    1. Crop to YOLO bbox
    2. Otsu threshold
    3. Morphological close (fill small holes)
    4. Keep ONLY the largest connected component — discards background blobs
    5. Find contour of that clean blob
    6. Fit min enclosing circle
    7. Contour points beyond outlier_thresh * median_radius = tab protrusion
    8. Centroid of those points → angle
    """
    H, W = img_bgr.shape[:2]
    x1 = max(0, int(bx));           y1 = max(0, int(by))
    x2 = min(W, int(bx + bw));      y2 = min(H, int(by + bh))
    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    # Step 2: Otsu threshold
    _, blob = cv2.threshold(gray, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 3: morphological close to fill holes inside lid
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    blob   = cv2.morphologyEx(blob, cv2.MORPH_CLOSE, kernel)

    # Step 4: keep only the largest connected component
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        blob, connectivity=8)
    if n_labels < 2:
        return None   # no foreground at all

    # Label 0 is background — find the largest foreground label
    areas = stats[1:, cv2.CC_STAT_AREA]   # skip background
    largest_label = int(np.argmax(areas)) + 1
    clean_blob = np.uint8(labels == largest_label) * 255

    # Step 5: contour of the clean blob
    contours, _ = cv2.findContours(clean_blob, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    if len(contour) < 10:
        return None

    # Step 6: fit minimum enclosing circle
    (fit_cx, fit_cy), fit_r = cv2.minEnclosingCircle(contour)
    pts   = contour[:, 0, :].astype(float)   # (N, 2)
    dists = np.sqrt((pts[:,0]-fit_cx)**2 + (pts[:,1]-fit_cy)**2)
    median_r = np.median(dists)

    # Step 7: outlier contour points = tab protrusion
    tab_mask = dists > outlier_thresh * median_r
    if tab_mask.sum() < 3:
        # Loosen threshold if nothing found
        tab_mask = dists > 1.02 * median_r
    if tab_mask.sum() < 3:
        return None

    tab_pts = pts[tab_mask]
    dx =  np.mean(tab_pts[:,0]) - fit_cx
    dy = -(np.mean(tab_pts[:,1]) - fit_cy)   # negate: image Y down

    if abs(dx) < 0.5 and abs(dy) < 0.5:
        return None

    return math.degrees(math.atan2(dy, dx)) % 360

In [ ]:
# ── Previous working method (57° mean, no polarity flip) ─────────────────────
def angle_contour_original(img_bgr, cx, cy, bx, by, bw, bh,
                            outlier_thresh=1.05):
    """Original contour deviation — the 57° mean version."""
    H, W = img_bgr.shape[:2]
    x1 = max(0, int(bx));           y1 = max(0, int(by))
    x2 = min(W, int(bx+bw));        y2 = min(H, int(by+bh))
    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0: return None
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, blob = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    blob    = cv2.morphologyEx(blob, cv2.MORPH_CLOSE, kernel)
    cnts, _ = cv2.findContours(blob, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts: return None
    contour = max(cnts, key=cv2.contourArea)
    if len(contour) < 10: return None
    (fit_cx, fit_cy), fit_r = cv2.minEnclosingCircle(contour)
    pts   = contour[:,0,:].astype(float)
    dists = np.sqrt((pts[:,0]-fit_cx)**2 + (pts[:,1]-fit_cy)**2)
    median_r = np.median(dists)
    tab_mask = dists > outlier_thresh * median_r
    if tab_mask.sum() < 3:
        tab_mask = dists > outlier_thresh * fit_r
    if tab_mask.sum() < 3: return None
    tab_pts = pts[tab_mask]
    dx =  np.mean(tab_pts[:,0]) - fit_cx
    dy = -(np.mean(tab_pts[:,1]) - fit_cy)
    if abs(dx)<0.5 and abs(dy)<0.5: return None
    return math.degrees(math.atan2(dy, dx)) % 360


# ── Benchmark both on GT val bboxes ──────────────────────────────────────────
errs_orig, errs_clean = [], []

for _, row in tqdm(gt_val.iterrows(), total=len(gt_val)):
    img = cv2.imread(os.path.join(IMG_DIR, row['image']))
    gt  = row['angle_deg']
    p_o = angle_contour_original(
            img, row['center_x'], row['center_y'],
            row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h'])
    p_c = angle_clean_contour(
            img, row['center_x'], row['center_y'],
            row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h'])
    errs_orig.append( circ_err(p_o if p_o is not None else 0.0, gt))
    errs_clean.append(circ_err(p_c if p_c is not None else 0.0, gt))

errs_orig  = np.array(errs_orig)
errs_clean = np.array(errs_clean)

print(f"{'Method':<40} {'Mean':>7} {'Median':>7} {'<20°':>7} {'<45°':>7}")
print('-'*67)
print(f"{'Original contour (57° baseline)':<40} "
      f"{errs_orig.mean():>7.1f} {np.median(errs_orig):>7.1f} "
      f"{(errs_orig<20).mean()*100:>6.1f}% {(errs_orig<45).mean()*100:>6.1f}%")
print(f"{'Clean LCC contour':<40} "
      f"{errs_clean.mean():>7.1f} {np.median(errs_clean):>7.1f} "
      f"{(errs_clean<20).mean()*100:>6.1f}% {(errs_clean<45).mean()*100:>6.1f}%")

In [ ]:
# ── Visual debug: show clean_blob vs original blob on 8 crops ────────────────
sample = gt_val.sample(8, random_state=9).reset_index(drop=True)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(sample.iterrows()):
    img  = cv2.imread(os.path.join(IMG_DIR, row['image']))
    H, W = img.shape[:2]
    x1 = max(0, int(row['bbox_x']));               y1 = max(0, int(row['bbox_y']))
    x2 = min(W, int(row['bbox_x']+row['bbox_w'])); y2 = min(H, int(row['bbox_y']+row['bbox_h']))
    crop = img[y1:y2, x1:x2].copy()
    vis  = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    # Rebuild clean blob for display
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, blob = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    blob    = cv2.morphologyEx(blob, cv2.MORPH_CLOSE, kernel)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(blob, 8)
    if n_labels >= 2:
        largest = int(np.argmax(stats[1:, cv2.CC_STAT_AREA])) + 1
        clean   = np.uint8(labels == largest) * 255
        noise   = blob & ~clean
        # Colour: clean blob = slight blue tint, discarded noise = red
        vis[clean > 0]  = (vis[clean > 0] * 0.6 + np.array([80,80,200]) * 0.4).astype(np.uint8)
        vis[noise > 0]  = [220, 50, 50]

        # Draw fitted circle on clean blob
        cnts, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            (fcx, fcy), fr = cv2.minEnclosingCircle(c)
            pts   = c[:,0,:].astype(float)
            dists = np.sqrt((pts[:,0]-fcx)**2+(pts[:,1]-fcy)**2)
            med_r = np.median(dists)
            cv2.circle(vis,(int(fcx),int(fcy)),int(med_r),(0,200,0),1)
            # Tab outlier points in yellow
            tab_m = dists > 1.05*med_r
            for p in pts[tab_m].astype(int):
                cv2.circle(vis, tuple(p), 2, (255,230,0), -1)

    # Draw arrows
    ch, cw = vis.shape[:2]
    centre = (cw//2, ch//2)
    L = int(min(cw,cw)*0.35)
    gt_a  = row['angle_deg']
    p_new = angle_clean_contour(img, row['center_x'], row['center_y'],
                                row['bbox_x'], row['bbox_y'],
                                row['bbox_w'], row['bbox_h'])
    p_old = angle_contour_original(img, row['center_x'], row['center_y'],
                                   row['bbox_x'], row['bbox_y'],
                                   row['bbox_w'], row['bbox_h'])

    for angle, color, thick in [
        (gt_a,  (255,220,0),  2),
        (p_old, (180,180,180),1),
        (p_new, (0,220,255),  2),
    ]:
        if angle is None: continue
        r  = math.radians(angle)
        ep = (centre[0]+int(L*math.cos(r)), centre[1]-int(L*math.sin(r)))
        cv2.arrowedLine(vis, centre, ep, color, thick, tipLength=0.25)

    axes[i].imshow(vis)
    e_new = circ_err(p_new, gt_a) if p_new else 999
    e_old = circ_err(p_old, gt_a) if p_old else 999
    color = 'green' if e_new<20 else ('orange' if e_new<45 else 'red')
    axes[i].set_title(
        f'GT:{gt_a:.0f}°  New:{p_new:.0f}° err={e_new:.0f}°\n(Old err={e_old:.0f}°)',
        fontsize=7, color=color)
    axes[i].axis('off')

plt.suptitle(
    'LCC fix: Blue=kept lid blob, Red=discarded noise\n'
    'Yellow dots=tab outlier pts, Yellow arrow=GT, Cyan=new pred, Grey=old pred',
    fontsize=9)
plt.tight_layout()
plt.savefig('/content/lcc_debug.png', dpi=150)
plt.show()

In [ ]:
# ── Error histograms ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, errs, label, color in [
    (axes[0], errs_orig,  f'Original (mean={errs_orig.mean():.1f}°)', 'steelblue'),
    (axes[1], errs_clean, f'LCC fix  (mean={errs_clean.mean():.1f}°)', 'seagreen')
]:
    ax.hist(errs, bins=18, color=color, edgecolor='k', alpha=0.85)
    ax.axvline(errs.mean(), color='red', ls='--', label=f'Mean {errs.mean():.1f}°')
    ax.axvline(np.median(errs), color='orange', ls=':',
               label=f'Median {np.median(errs):.1f}°')
    ax.axvline(45, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Angle error (°)')
    ax.set_title(label)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Count')
plt.tight_layout()
plt.savefig('/content/lcc_histogram.png', dpi=150)
plt.show()

In [ ]:
# ── Apply to YOLO detections + save final metrics ─────────────────────────────
DIST_THRESH  = 30.0
FINAL_CSV    = '/content/final_predictions.csv'
METRICS_JSON = '/content/metrics.json'

yolo_df  = pd.read_csv('/content/yolo_predictions.csv')
pred_val = yolo_df[yolo_df['image'].isin(val_set)].reset_index(drop=True)

final_rows = []
for _, row in tqdm(pred_val.iterrows(), total=len(pred_val)):
    img   = cv2.imread(os.path.join(IMG_DIR, row['image']))
    angle = angle_clean_contour(
                img, row['center_x'], row['center_y'],
                row['bbox_x'], row['bbox_y'],
                row['bbox_w'], row['bbox_h'])
    if angle is None: angle = 0.0
    final_rows.append({'image': row['image'],
                       'center_x': row['center_x'],
                       'center_y': row['center_y'],
                       'angle_deg': angle})

final_df = pd.DataFrame(final_rows)
final_df.to_csv(FINAL_CSV, index=False)